In [43]:
!pip install tensorflow==2.12.0

  Using cached tensorflow-2.12.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.4 kB)
  Using cached keras-2.12.0-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached tensorflow-2.12.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (586.0 MB)
Using cached keras-2.12.0-py2.py3-none-any.whl (1.7 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-decision-forests 1.11.0 requires tensorflow==2.18.0, but you have tensorflow 2.12.0 which is incompatible.
tf-keras 2.18.0 requires tensorflow<2.19,>=2.18, but you have tensorflow 2.12.0 which is incompatible.
tensorflow-text 2.18.1 requires tensorflow<2.19,>=2.18.0, but you have tensorflow 2.12.0 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.0 which is incompatible.


In [1]:
!pip install keras

In [2]:
import tensorflow as tf

print(tf.__version__)

2.12.0


In [5]:
!pip install keras-rl2==1.0.5

  Using cached keras_rl2-1.0.5-py3-none-any.whl.metadata (304 bytes)
Using cached keras_rl2-1.0.5-py3-none-any.whl (52 kB)


In [95]:
!pip install gym==0.21.0

  Using cached gym-0.21.0.tar.gz (1.5 MB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [94]:
!pip uninstall gym==0.26.0

Found existing installation: gym 0.26.0
Uninstalling gym-0.26.0:
  Would remove:
    /usr/local/lib/python3.11/dist-packages/gym-0.26.0.dist-info/*
    /usr/local/lib/python3.11/dist-packages/gym/*
Proceed (Y/n)? Y
  Successfully uninstalled gym-0.26.0


In [7]:
!pip install numpy

In [99]:
import gym
import random
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers.legacy import Adam
from rl.agents.dqn import DQNAgent
from rl.policy import BoltzmannQPolicy
from rl.memory import SequentialMemory
from rl.core import Processor

# Wrapper برای سازگاری gym 0.26+ با keras-rl2
class GymV26CompatibleEnv(gym.Wrapper):
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        done = terminated or truncated
        return obs, reward, done, info

    def reset(self, **kwargs):
        return self.env.reset(**kwargs)[0]

# Processor برای اصلاح ورودی‌ها
class GymProcessor(Processor):
    def process_observation(self, observation):
        return np.array(observation)

    def process_state_batch(self, batch):
        return batch

# ساخت محیط و wrap کردن آن
env = gym.make('CartPole-v1', render_mode="human")
env = GymV26CompatibleEnv(env)

states = env.observation_space.shape[0]  # 4
actions = env.action_space.n              # 2

print(f"States: {states}, Actions: {actions}")

# ساخت مدل
model = Sequential()
model.add(Flatten(input_shape=(1, states)))
model.add(Dense(24, activation='relu'))
model.add(Dense(24, activation='relu'))
model.add(Dense(actions, activation='linear'))
model.summary()

# ساخت عامل DQN
memory = SequentialMemory(limit=50000, window_length=1)
policy = BoltzmannQPolicy()

dqn = DQNAgent(model=model, memory=memory, policy=policy,
               nb_actions=actions,
               nb_steps_warmup=10,
               target_model_update=1e-2,
               processor=GymProcessor())

dqn.compile(Adam(learning_rate=1e-3), metrics=['mae'])

# آموزش
dqn.fit(env, nb_steps=20000, visualize=False, verbose=1)

# تست
dqn.test(env, nb_episodes=5, visualize=False)

# بستن محیط
env.close()

States: 4, Actions: 2
Model: "sequential_16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_13 (Flatten)        (None, 4)                 0         
                                                                 
 dense_48 (Dense)            (None, 24)                120       
                                                                 
 dense_49 (Dense)            (None, 24)                600       
                                                                 
 dense_50 (Dense)            (None, 2)                 50        
                                                                 
Total params: 770
Trainable params: 770
Non-trainable params: 0
_________________________________________________________________
Training for 20000 steps ...
Interval 1 (0 steps performed)


/usr/local/lib/python3.11/dist-packages/keras/engine/training_v1.py:2359: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


   10/10000 [..............................] - ETA: 3:27 - reward: 1.0000

/usr/local/lib/python3.11/dist-packages/rl/memory.py:37: UserWarning: Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!
  warnings.warn('Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!')
/usr/local/lib/python3.11/dist-packages/keras/engine/training_v1.py:2359: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


   21/10000 [..............................] - ETA: 21:43 - reward: 1.0000

/usr/local/lib/python3.11/dist-packages/rl/memory.py:37: UserWarning: Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!
  warnings.warn('Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!')


10000/10000 [==============================] - 210s 21ms/step - reward: 1.0000
89 episodes - episode_reward: 111.719 [9.000, 365.000] - loss: 2.251 - mae: 19.670 - mean_q: 39.895

Interval 2 (10000 steps performed)
10000/10000 [==============================] - 206s 21ms/step - reward: 1.0000
done, took 415.550 seconds
Testing for 5 episodes ...
Episode 1: reward: 218.000, steps: 218
Episode 2: reward: 215.000, steps: 215
Episode 3: reward: 236.000, steps: 236
Episode 4: reward: 231.000, steps: 231
Episode 5: reward: 267.000, steps: 267
